In [1]:
%pip install opencv-python opencv-python-headless
%pip install opencv-contrib-python
%pip install numpy
%pip install matplotlib



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: C:\Users\aditi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/46.2 MB ? eta -:--:--
     ---------------------------------------- 0.1/46.2 MB 1.5 MB/s eta 0:00:31
     ---------------------------------------- 0.2/46.2 MB 2.8 MB/s eta 0:00:17
     ---------------------------------------- 0.3/46.2 MB 2.8 MB/s eta 0:00:17
      --------------------------------------- 0.6/46.2 MB 3.3 MB/s eta 0:00:14
      --------------------------------------- 0.8/46.2 MB 3.6 MB/s eta 0:00:13
      --------------------------------------- 1.0/46.2 MB 3.9 MB/s eta 0:00:12
     - -------------------------------------- 1.2/46.2 MB 3.9 MB/s eta 0:00:12
     - -------------------------------------- 1.4/46.2 MB 3.8 MB/s eta 0:00:12
     - -------------------------------------- 1.5/46.2 MB 3.7 MB/s eta 0:00:13
     - -------------------------------------- 1.6/46.2 MB 3.4 MB/s eta 0:00:14
     - -------------------------------------- 1.8/46.2 MB 3.6 MB/s eta 0:00:13
     - -------------------------------------- 1.8/46.2 MB 3


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: C:\Users\aditi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: C:\Users\aditi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
     ---------------------------------------- 0.1/8.1 MB 2.6 MB/s eta 0:00:04
     - -------------------------------------- 0.2/8.1 MB 3.1 MB/s eta 0:00:03
     - -------------------------------------- 0.3/8.1 MB 2.5 MB/s eta 0:00:04
     -- ------------------------------------- 0.4/8.1 MB 2.6 MB/s eta 0:00:03
     -- ------------------------------------- 0.4/8.1 MB 2.6 MB/s eta 0:00:03
     -- ------------------------------------- 0.4/8.1 MB 2.6 MB/s eta 0:00:03
     -- ------------------------------------- 0.4/8.1 MB 2.6 MB/s eta 0:00:03
     -- ------------------------------------- 0.4/8.1 MB 2.6 MB/s eta 0:00:03
     -- ------------------------------------- 0.4/8.1 MB 2.6 MB/s eta 0:00:03
     -- ------------------------------------- 0.5/8.1 MB 909.8 kB/s eta 0:00:09
     -- ------------------------------------- 0.5/8.1 MB 964.2 kB/s eta 0:00:08
     --- ------------------------------------ 0.7/8.1 MB 1.2 MB/s e


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: C:\Users\aditi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import cv2
import numpy as np

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)


OpenCV version: 4.11.0
NumPy version: 2.2.6


In [ ]:
import cv2
import numpy as np

# Open the default webcam
cap = cv2.VideoCapture(0)

# Check if camera opened successfully
if not cap.isOpened():
    print("Error: Could not open camera.")
else:
    print("Camera opened successfully.")

# Capture and save a frame
ret, frame = cap.read()
if ret:
    cv2.imwrite("frame_original.png", frame)
    print("Saved original frame.")
else:
    print("Failed to capture frame.")

# Release the camera
cap.release()
# Load the saved image
frame = cv2.imread("frame_original.png")

# Convert to grayscale
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
cv2.imwrite("frame_gray.png", gray)
clahe1 = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(8, 8))
clahe2 = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
enhanced = clahe1.apply(gray)
enhanced1 = clahe2.apply(gray)
cv2.imwrite("frame_enhanced.png", enhanced)
cv2.imwrite("frame_enhanced1.png", enhanced1)
denoised = cv2.fastNlMeansDenoising(enhanced, h=10)
cv2.imwrite("frame_denoised.png", denoised)

from matplotlib import pyplot as plt

def show_img(img_path, title=""):
    img = cv2.imread(img_path)        
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

show_img("frame_original.png", "Original Frame")
show_img("frame_gray.png", "Grayscale")
show_img("frame_enhanced.png", "Enhanced")
show_img("frame_enhanced1.png", "Enhanced1")
show_img("frame_denoised.png", "Denoised")

Camera opened successfully.


#### Using EAST for Bounding Boxes

In [3]:
import cv2
import numpy as np

# Load pretrained EAST model
east_model = 'frozen_east_text_detection.pb'
net = cv2.dnn.readNet(east_model)

# Set input size and thresholds
input_width = 320
input_height = 320
conf_threshold = 0.5
nms_threshold = 0.4

def decode_predictions(scores, geometry, score_thresh):
    detections = []
    confidences = []

    height, width = scores.shape[2:4]

    for y in range(height):
        scores_data = scores[0, 0, y]
        x0_data = geometry[0, 0, y]
        x1_data = geometry[0, 1, y]
        x2_data = geometry[0, 2, y]
        x3_data = geometry[0, 3, y]
        angles_data = geometry[0, 4, y]

        for x in range(width):
            score = scores_data[x]
            if score < score_thresh:
                continue

            offset_x, offset_y = x * 4.0, y * 4.0
            angle = angles_data[x]
            cos = np.cos(angle)
            sin = np.sin(angle)

            h = x0_data[x] + x2_data[x]
            w = x1_data[x] + x3_data[x]

            end_x = offset_x + cos * x1_data[x] + sin * x2_data[x]
            end_y = offset_y - sin * x1_data[x] + cos * x2_data[x]
            start_x = end_x - w
            start_y = end_y - h

            detections.append((int(start_x), int(start_y), int(w), int(h)))
            confidences.append(float(score))

    return detections, confidences

# Open webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    orig = frame.copy()
    H, W = frame.shape[:2]

    # Resize and prepare blob
    blob = cv2.dnn.blobFromImage(frame, 1.0, (input_width, input_height),
                                 (123.68, 116.78, 103.94), swapRB=True, crop=False)
    net.setInput(blob)
    scores, geometry = net.forward(["feature_fusion/Conv_7/Sigmoid",
                                    "feature_fusion/concat_3"])

    # Decode and filter boxes
    boxes, confidences = decode_predictions(scores, geometry, conf_threshold)
    indices = cv2.dnn.NMSBoxes(boxes, confidences, conf_threshold, nms_threshold)

    # Scaling factors
    r_w = W / float(input_width)
    r_h = H / float(input_height)

    for i in indices:
        i = i[0] if isinstance(i, (list, np.ndarray)) else i
        x, y, w, h = boxes[i]

        x = int(x * r_w)
        y = int(y * r_h)
        w = int(w * r_w)
        h = int(h * r_h)

        cv2.rectangle(orig, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow("Text Detection", orig)

    if cv2.waitKey(1) & 0xFF == 27:  # Press Esc to quit
        break

cap.release()
cv2.destroyAllWindows()


### Using PaddleOCR for text detection

In [4]:
%pip install paddleocr
%pip install "paddlepaddle==2.5.2" -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html

     ---------------------------------------- 0.0/63.3 kB ? eta -:--:--
     ---------------------------------------- 63.3/63.3 kB 3.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/161.8 kB ? eta -:--:--
     -------------- ------------------------ 61.4/161.8 kB 1.6 MB/s eta 0:00:01
     ------------------------------------ - 153.6/161.8 kB 1.5 MB/s eta 0:00:01
     -------------------------------------- 161.8/161.8 kB 1.6 MB/s eta 0:00:00
     ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
     -- ------------------------------------- 0.1/1.7 MB 3.6 MB/s eta 0:00:01
     ----- ---------------------------------- 0.2/1.7 MB 3.5 MB/s eta 0:00:01
     ------- -------------------------------- 0.3/1.7 MB 2.0 MB/s eta 0:00:01
     --------- ------------------------------ 0.4/1.7 MB 1.9 MB/s eta 0:00:01
     ----------- ---------------------------- 0.5/1.7 MB 1.9 MB/s eta 0:00:01
     ------------- -------------------------- 0.6/1.7 MB 2.1 MB/s eta 0:00

  DEPRECATION: GPUtil is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\aditi\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python310\\site-packages\\~umpy.libs\\libscipy_openblas64_-13e2df515630b4a41f92893938845698.dll'
Check the permissions.


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: C:\Users\aditi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Looking in links: https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
     ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
     --------------------------------------- 0.0/72.0 MB 660.6 kB/s eta 0:01:49
     --------------------------------------- 0.0/72.0 MB 660.6 kB/s eta 0:01:49
     --------------------------------------- 0.1/72.0 MB 901.1 kB/s eta 0:01:20
     --------------------------------------- 0.1/72.0 MB 901.1 kB/s eta 0:01:20
     --------------------------------------- 0.1/72.0 MB 901.1 kB/s eta 0:01:20
     --------------------------------------- 0.1/72.0 MB 901.1 kB/s eta 0:01:20
     --------------------------------------- 0.1/72.0 MB 901.1 kB/s eta 0:01:20
     --------------------------------------- 0.1/72.0 MB 901.1 kB/s eta 0:01:20
     --------------------------------------- 0.1/72.0 MB 901.1 kB/s eta 0:01:20
     --------------------------------------- 0.1/72.0 MB 901.1 kB/s eta 0:01:20
     --------------------------------------- 0.2

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\aditi\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python310\\site-packages\\paddle\\include\\paddle\\phi\\kernels\\fusion\\cutlass\\memory_efficient_attention\\iterators\\epilogue_predicated_tile_iterator.h'


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: C:\Users\aditi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
